# Get to Know a Dataset: Korean Variant Archive 3 (KOVA3)

This notebook serves as a guided tour of the [Korean Variant Archive 3 (KOVA3)](https://registry.opendata.aws/kova3) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

KOVA3 is a population-scale whole-genome reference for Korean and East Asian genomics, built from 11,008 Korean genomes jointly genotyped on GRCh38. It is released in two tiers:

- **Open tier (`kova3-open`)**, covered by this notebook: aggregate allele-frequency data under CC BY 4.0. No registration, no AWS account, no agreement.
- **Controlled-access tier (`kova3-controlled`)**: participant-level FASTQ, CRAM, per-sample gVCF and the genotyped multi-sample VCF, available to approved academic researchers under the KOVA3 Data Use Agreement. See the [data access policy](https://github.com/KuChoiLab/kova3/blob/main/docs/data-access.md).

Most users need only the open tier. Frequency filtering, ACMG/AMP population-frequency evidence, carrier-frequency estimation and comparative population genetics are all answered here.

> **Draft note for the AWS Open Data team.** The S3 buckets are not yet created, so the code cells below carry no outputs. Bucket names and the release tag are written as the values we intend to use; we will run the notebook end to end and commit it with outputs cleared once the buckets exist.

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The open-tier bucket separates documentation, metadata and data at the top level, and versions every release with a Hive-style `release=` prefix so that an analysis can pin the exact release it used:

```
s3://kova3-open/
├── README.md
├── LICENSE                       # CC BY 4.0
├── CHANGELOG.md
├── docs/                         # Snapshot of the documentation repository at release time
├── metadata/
│   ├── release=v3.0.0/           # manifest.json, cohort and QC summaries
│   └── schemas/                  # Parquet, Hail and VCF header schemas
└── data/
    └── release=v3.0.0/
        ├── sites_vcf/            # Chromosome-sharded, bgzip + tabix
        ├── parquet/              # Partitioned by chromosome and position bin
        ├── hail/                 # Prebuilt Hail Table
        └── callability/          # Allele number and call rate per site
```

Published paths are immutable. A correction produces a new `release=` prefix rather than rewriting an existing one, so a pipeline pinned to `v3.0.0` keeps returning the same answer. There is deliberately no "latest" alias.

The three representations under `data/` hold the same allele-frequency callset in three shapes, each suited to a different access pattern: indexed VCF for pulling one interval, Parquet for querying many variants at once, and a Hail Table for genome-wide work.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# polars >= 1.30.0
# pyarrow >= 16.0.0
# matplotlib >= 3.10.3
#
# The bcftools examples below call the command-line tool (>= 1.19, built with libcurl
# so that it can read https:// URLs). Install it with conda: conda install -c bioconda bcftools

# Built-ins
import subprocess
from pprint import pprint

# Installed libraries
import boto3, polars, matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

The open tier is a public bucket, so requests do not need to be signed. Below we define the bucket and release we will use throughout, then list the top-level prefixes.

In [ ]:
# Location of the open-tier bucket and the release this notebook is pinned to
BUCKET  = "kova3-open"
REGION  = "ap-northeast-2"
RELEASE = "v3.0.0"

# Public bucket: set the signature version to unsigned so no credentials are needed
s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))

# Print the top-level prefixes and objects
response = s3.list_objects_v2(Bucket=BUCKET, Delimiter="/")
for item in response.get("CommonPrefixes", []):
    print(item["Prefix"])
for item in response.get("Contents", []):
    print(item["Key"])

Looking inside `data/`, each release sits under its own `release=` prefix.

In [ ]:
# List the releases available in the bucket
for item in s3.list_objects_v2(Bucket=BUCKET, Prefix="data/", Delimiter="/")["CommonPrefixes"]:
    print(item["Prefix"])

Within a release, the four representations of the callset sit side by side. Listing the sites-only VCF prefix shows one bgzip-compressed shard and one tabix index per contig.

In [ ]:
# List the four representations within the pinned release
for item in s3.list_objects_v2(Bucket=BUCKET, Prefix=f"data/release={RELEASE}/", Delimiter="/")["CommonPrefixes"]:
    print(item["Prefix"])

print()

# List the first few sites-only VCF shards and their indexes
for item in s3.list_objects_v2(Bucket=BUCKET, Prefix=f"data/release={RELEASE}/sites_vcf/", MaxKeys=6)["Contents"]:
    print(f"{item['Key']}  ({item['Size'] / 1e6:.1f} MB)")

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats?

Every format here is a community standard readable with open-source software. No proprietary reader is required.

| Format | What it holds | Why this format |
|---|---|---|
| bgzip VCF + tabix index (`.vcf.gz`, `.vcf.gz.tbi`) | Sites-only callset: one row per variant with AC, AN, AF, homozygote count, call rate and filter status | The lingua franca of variant data. bgzip is block-compressed, so a tabix index lets a client fetch only the blocks covering a genomic interval over HTTP range requests, without downloading the shard. |
| Apache Parquet | The same allele-frequency table, partitioned by `chromosome` and `position_bin` | Columnar and predicate-friendly. Amazon Athena queries it in place, and partition pruning plus column projection keep a gene-panel lookup to a few megabytes scanned. |
| Hail Table (`.ht`) | The same callset in Hail's native on-disk format | Users doing genome-wide work in Hail would otherwise spend time and money importing and transforming the VCF first. Shipping the table prebuilt removes that step. |
| TSV and Parquet | Aggregate cohort metadata, QC summaries | Stored apart from the genomic data so users can inspect cohort composition without scanning the callset. |
| JSON | Schemas and the release manifest with per-object SHA-256 checksums | Machine-readable, so a pipeline can verify a download and discover the schema without parsing a VCF header. |

**One KOVA3-specific point worth knowing.** The callset is *sites-only*: it carries no per-sample genotype columns, no `FORMAT` fields and no participant-level metadata. An `AF` of 0 therefore means "not observed in this cohort", which is not the same as "not present in Koreans". To tell a truly absent variant from a poorly covered one, read the callability resources under `callability/`, which give allele number and call rate per site. This distinction matters when the absence of a variant is itself the evidence you are relying on.

**AWS services that fit these formats.** Amazon Athena and Amazon EMR (Spark, Hail) read the Parquet and Hail layers directly. Amazon EC2 and AWS Lambda can stream intervals from the indexed VCF. Nothing in the open tier requires an AWS account: the bucket is public and the program covers egress.

### Q: Can you show us an example of downloading and loading data from your dataset?

Two ways in, depending on what you are after.

First, a single gene. Here we stream the *PCSK9* locus straight out of the indexed sites-only VCF with `bcftools`. Only the compressed blocks covering the interval are transferred, so this costs a few hundred kilobytes rather than the whole shard.

In [ ]:
# Stream one gene region directly from S3. bcftools reads the tabix index over https
# and issues range requests for just the blocks that overlap the interval.
VCF_URL = f"https://{BUCKET}.s3.{REGION}.amazonaws.com/data/release={RELEASE}/sites_vcf/kova3.chr1.sites.vcf.gz"
PCSK9 = "chr1:55039447-55064852"   # PCSK9 gene body, GRCh38

query = subprocess.run(
    ["bcftools", "query", "-r", PCSK9,
     "-f", "%CHROM\t%POS\t%REF\t%ALT\t%INFO/AC\t%INFO/AN\t%INFO/AF\t%INFO/nhomalt\t%FILTER\n",
     VCF_URL],
    capture_output=True, text=True, check=True)

pcsk9 = polars.read_csv(
    query.stdout.encode(), separator="\t", has_header=False,
    new_columns=["chrom", "pos", "ref", "alt", "AC", "AN", "AF", "nhomalt", "filter"])

print(f"{pcsk9.height} variants in PCSK9")
pcsk9.head()

Second, many variants at once. For a gene panel, a variant list, or anything genome-wide, read the Parquet layer instead. Partitioning by chromosome and position bin means a bounded query touches only the files it needs.

In [ ]:
# Read one position bin of the Parquet layer. Partition columns are part of the path,
# so filtering on chromosome and position_bin prunes files before any bytes are read.
parquet_uri = f"s3://{BUCKET}/data/release={RELEASE}/parquet/chromosome=chr1/position_bin=005/"

chr1_bin = polars.read_parquet(
    parquet_uri,
    storage_options={"aws_region": REGION, "aws_skip_signature": "true"})

print(chr1_bin.schema)
chr1_bin.head()

The Parquet layer can also be queried with Amazon Athena, which is usually the cheapest way to look up a gene panel or a list of a few thousand variants. Athena reads the public bucket directly; you register the table once in your own AWS account using the `CREATE EXTERNAL TABLE` statement published under `metadata/schemas/`, and the table and its query charges stay in your account. Partition pruning on `chromosome` and `position_bin` keeps the bytes scanned small.

```sql
SELECT chromosome, pos, ref, alt, ac, an, af, nhomalt
FROM   kova3.sites_v3_0_0
WHERE  chromosome = 'chr1'
  AND  pos BETWEEN 55039447 AND 55064852
  AND  af >= 0.01;
```

### Q: A picture is worth a thousand words. Show us a visual from your dataset.

The site frequency spectrum is the first thing most people want to see from a population callset: it says how much of the catalogue is rare. We build it from one chromosome's Parquet partition, which is enough to show the shape without reading the genome.

In [ ]:
# Read the allele-frequency column for one chromosome
chr1 = polars.read_parquet(
    f"s3://{BUCKET}/data/release={RELEASE}/parquet/chromosome=chr1/",
    columns=["af", "ac", "an"],
    storage_options={"aws_region": REGION, "aws_skip_signature": "true"})

# Keep sites with a usable allele number, then bin allele frequency on a log scale
observed = chr1.filter((polars.col("an") > 0) & (polars.col("af") > 0))
print(f"{observed.height:,} variants with AF > 0 on chr1")

plt.figure(figsize=(11, 6), dpi=100, facecolor="white")
plt.hist(observed["af"], bins=60, log=True, color="#3498db", edgecolor="white", linewidth=1.0)
plt.xscale("log")
plt.title("KOVA3 site frequency spectrum, chr1", fontsize=15, pad=16, fontweight="bold")
plt.xlabel("Alternate allele frequency", fontsize=12, labelpad=10)
plt.ylabel("Number of variants", fontsize=12, labelpad=10)
plt.grid(True, linestyle="--", alpha=0.3, color="gray")
ax = plt.gca()
ax.set_facecolor("#f8f9fa")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

The spectrum is dominated by rare variation, as expected for a sample of this size, with singletons and doubletons making up the bulk of the catalogue. That shape is what makes the next question interesting: the variants that matter most for clinical interpretation are the ones sitting at the other end.

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**How many variants are common in Koreans but look rare in the reference databases clinical laboratories actually use?**

This is the question that motivates KOVA3. East Asians account for only 2.78% of gnomAD v4, and Koreans are genetically distinct from Chinese and Japanese populations, so a variant that is unremarkable in Korea can appear rare in a broader East Asian aggregate. When that happens, a benign polymorphism survives rare-disease filtering and lands in a clinician's candidate list.

The ACMG/AMP framework gives population frequency real weight: BA1 is stand-alone benign evidence when a variant is too common to cause a rare disease, and BS1 is strong benign evidence when its frequency exceeds what the disorder's prevalence allows. Both depend on having a frequency from a matched population.

Below we count the variants where KOVA3 and a broader reference disagree enough to change that call.

In [ ]:
# Load the Korean allele frequencies for one chromosome
kova3 = polars.read_parquet(
    f"s3://{BUCKET}/data/release={RELEASE}/parquet/chromosome=chr1/",
    columns=["chrom", "pos", "ref", "alt", "af", "an"],
    storage_options={"aws_region": REGION, "aws_skip_signature": "true"}
).rename({"af": "af_kova3"})

# Join against a broader East Asian reference keyed on chrom/pos/ref/alt.
# Substitute the reference of your choice; the join key is the only requirement.
reference = load_reference_allele_frequencies(chrom="chr1")   # user-supplied
compared = kova3.join(reference, on=["chrom", "pos", "ref", "alt"], how="inner")

# Variants that pass a 5% filter in Koreans but would not have been filtered on the broader reference
discordant = compared.filter(
    (polars.col("af_kova3") >= 0.05) & (polars.col("af_reference") < 0.01))

print(f"{discordant.height:,} variants on chr1 are common in Koreans (AF >= 5%) "
      f"but rare in the broader reference (AF < 1%)")
discordant.sort("af_kova3", descending=True).head(10)

Every row in that table is a variant a Korean patient is likely to carry, that a frequency filter built on a non-matched reference would have let through. In our own rare-disease work this is where KOVA3 earns its place: the candidate list gets shorter, and the variants it removes are the ones that were never plausible.

The same comparison run the other way is worth doing too. Variants that are common in the broader reference but rare in Koreans are a reminder that a population-matched frequency cuts both ways, and that BA1 or BS1 applied from the wrong population can be wrong in either direction.

### Q: What is one unanswered question that you think could be answered using these data?

**How much does a population-matched frequency filter actually change the diagnostic outcome for Korean rare-disease cases, and does the effect differ by disease area?**

We can show that the candidate list gets shorter. What we cannot yet show, from aggregate data alone, is how often that shortening changes a diagnosis: how many cases reach a confident answer that would otherwise have stalled, and whether the gain concentrates in particular phenotype groups or inheritance patterns.

**The challenge.** Take a set of Korean rare-disease cases you already work with. Filter the candidate variants twice, once against a broad reference and once against KOVA3, and report three numbers: the change in candidate burden per case, the number of variants reclassified under BA1 or BS1, and the number of cases whose top candidate changed. Post what you find as an issue on the [KOVA3 repository](https://github.com/KuChoiLab/kova3/issues).

**Advice for anyone taking this on.** Read the callability resources alongside the frequencies. A variant absent from KOVA3 is only evidence of rarity where the allele number at that site is high, and treating low-coverage sites as absences is the mistake most likely to distort your result. If your analysis needs individual genotypes, haplotypes or read-level evidence, the controlled-access tier exists for exactly that: the application process is described in the [data access policy](https://github.com/KuChoiLab/kova3/blob/main/docs/data-access.md).

## Where to go next

- [KOVA3 documentation](https://github.com/KuChoiLab/kova3): cohorts and consent basis, joint-genotyping methods and QC, the INFO field data dictionary, Parquet and Hail schemas, the versioning policy and how to cite.
- [Data access policy](https://github.com/KuChoiLab/kova3/blob/main/docs/data-access.md) for the controlled-access tier.
- Further tutorials in the [tutorials directory](https://github.com/KuChoiLab/kova3/tree/main/tutorials): annotating a patient VCF, querying with Athena, and genome-wide analysis in Hail on Amazon EMR.

Questions about the data, or something that looks wrong? Open an issue on the repository.